In [11]:
from pathlib import Path

try:
    from google.colab import drive
except ModuleNotFoundError:
    print("Local environment detected; Google Drive mount skipped.")
else:
    if Path("/content/drive/MyDrive").is_dir():
        print("Google Drive is already mounted.")
    else:
        drive.mount("/content/drive")

Local environment detected; Google Drive mount skipped.


In [12]:
# -*- coding: utf-8 -*-
import copy
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        *cwd.parents,
        cwd / "Grad_Research",
        cwd.parent,
        Path("/content/Grad_Research"),
        Path(
            "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
        ),
        Path("/content/drive/MyDrive/Grad_Research"),
        Path(
            "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
        ),
        Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
        Path(
            "/content/drive/MyDrive/Colab Notebooks/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
        ),
    ])

    for root in candidates:
        if (root / "EXP_v2" / "data" / "theta_true").is_dir():
            return root

    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "EXP_v2" / "data" / "theta_true").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root containing EXP_v2/data/theta_true. "
        "In Colab, place the repository at MyDrive/Grad_Research or /content/Grad_Research."
    )


ROOT = find_project_root()
MODEL_DIR = ROOT / "EXP_v2" / "models"
RESULTS_DIR = ROOT / "EXP_v2" / "results"

print(f"Device      : {device}")
print(f"Project root: {ROOT}")
print(f"Model dir   : {MODEL_DIR}")
print(f"Results dir : {RESULTS_DIR}")

Device      : mps
Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Model dir   : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v2/models
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v2/results


In [13]:
@dataclass
class Config:
    # Network
    input_size:    int   = 1
    first_hidden:  int   = 50
    second_hidden: int   = 30
    dropout_rate:  float = 0.0

    # Training
    test_length:         int   = 10
    gamma:               float = 0.1
    memory_capacity:     int   = 1000
    epsilon:             float = 0.1
    batch_size:          int   = 128
    q_network_iteration: int   = 40
    learning_rate:       float = 1e-3
    training_size:       int   = 1000
    validation_size:     int   = 200
    validation_interval: int   = 50
    seed:                int   = 20260430
    validation_seed:     int   = 20260431
    test_seed:           int   = 20260432

    # Bank / prior
    bank_id:   int = 1
    prior:     str = "uniform"  # "normal" | "uniform"
    n_items:   int = 100


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [14]:
from typing import Any, cast
from scipy.optimize import minimize_scalar


# 2PL formulation (drops the guessing parameter c).
def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    p = 1.0 / (1.0 + np.exp(-D * a * (theta - b)))
    resp = (np.random.random(size=p.shape) <= p).astype(int)
    return resp


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    p = 1.0 / (1.0 + np.exp(-D * a * (theta - b)))
    return D**2 * a**2 * p * (1.0 - p)


def MLE(item_paras, resp, D=1):
    a = item_paras[:, 0]
    b = item_paras[:, 1]

    def mins_likelihood(x):
        logl = 0
        for i in range(len(resp)):
            p = 1.0 / (1.0 + np.exp(-D * a[i] * (x - b[i])))
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp[i] * np.log(p) + (1 - resp[i]) * np.log(1 - p)
        return logl

    result = cast(Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded"))
    return np.array(result.x).reshape(1,)


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = 1.0 / (1.0 + np.exp(-D * a[i] * (x - b[i])))
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        result = cast(Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded"))
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)


def Apply_Positive_Constraint(model, min_value=0.0):
    for param in model.parameters():
        param.data = torch.clamp(param.data, min=min_value)

In [15]:
class Net(nn.Module):
    def __init__(self, input_size, first_hidden, second_hidden, action_space, dropout_rate):
        super(Net, self).__init__()
        self.fc1     = nn.Linear(input_size, first_hidden)
        self.fc2     = nn.Linear(first_hidden, second_hidden)
        self.out     = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(self.fc1(x))
        x = F.relu(x)
        x = self.dropout(self.fc2(x))
        x = F.relu(x)
        return self.out(x)

    def initialize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)

In [16]:
def Choose_Action(item_id, state, epsilon):
    if np.random.rand() >= epsilon:
        state_t   = torch.unsqueeze(torch.FloatTensor(state), 0).to(device)
        item_id_t = torch.from_numpy(item_id).to(device).long()
        action_value = eval_net(state_t)
        action_value[:, item_id_t] = torch.zeros(item_id_t.shape).to(device)
        action = torch.max(action_value, -1)[1].cpu().numpy()
    else:
        if any(item_id):
            action = np.random.choice(np.delete(np.arange(action_space), item_id)).astype("int64").reshape(1,)
        else:
            action = np.random.choice(np.arange(action_space)).astype("int64").reshape(1,)
    return action


def Choose_Action_Test(item_id, state):
    state_t      = torch.FloatTensor(state.swapaxes(0, 1)).to(device)
    action_value = eval_net(state_t).detach().cpu().numpy()
    if item_id.shape[0] > 0:
        action_value[
            np.tile(np.arange(item_id.shape[1])[np.newaxis, :], (item_id.shape[0], 1)),
            item_id,
        ] = np.zeros(item_id.shape)
    return action_value.argmax(axis=1)

In [17]:
def TRAIN(
    cfg,
    training_theta,
    validation_theta,
    validation_initial_theta,
    validation_response_matrix,
):
    best_valid = None
    best_state = None

    loss_func = nn.MSELoss()
    eval_net.train()
    optimizer = optim.Adam(eval_net.parameters(), lr=cfg.learning_rate)

    memory         = np.zeros((cfg.memory_capacity, cfg.input_size * 2 + 2))
    memory_counter = 0
    learn_step_counter = 0

    for j in range(cfg.training_size):
        state   = np.concatenate((np.zeros(cfg.input_size - 1), np.random.rand(1) - 0.5))
        item_id = np.array([]).astype("int64")
        resp    = np.array([]).astype("int64")

        for i in range(cfg.test_length):
            action  = Choose_Action(item_id, state, cfg.epsilon)
            item_id = np.concatenate((item_id, action))
            resp    = np.concatenate((resp, RESPOND(item_bank[action], training_theta[j])))
            reward  = FI(item_bank[action,], training_theta[j])

            if len(np.unique(resp)) == 1:
                if resp[-1] == 1:
                    next_state = np.array([state[-1] + (item_bank[:, 1].max() - state[-1]) / 2])
                else:
                    next_state = np.array([state[-1] - (state[-1] - item_bank[:, 1].min()) / 2])
            else:
                next_state = MLE(item_bank[item_id,], resp)
            if cfg.input_size > 1:
                next_state = np.concatenate((state[-(cfg.input_size - 1):], next_state))

            memory[memory_counter % cfg.memory_capacity, :] = np.hstack((state, action, reward, next_state))
            memory_counter += 1
            state = next_state

            if memory_counter >= cfg.batch_size:
                batch_memory     = memory[np.random.choice(min(memory_counter, cfg.memory_capacity), cfg.batch_size), :]
                batch_state      = torch.FloatTensor(batch_memory[:, :cfg.input_size]).to(device)
                batch_action     = torch.LongTensor(batch_memory[:, cfg.input_size:cfg.input_size + 1].astype(int)).to(device)
                batch_reward     = torch.FloatTensor(batch_memory[:, cfg.input_size + 1:cfg.input_size + 2]).to(device)
                batch_next_state = torch.FloatTensor(batch_memory[:, -cfg.input_size:]).to(device)

                q_eval   = eval_net(batch_state).gather(1, batch_action)
                q_next   = target_net(batch_next_state).detach()
                q_target = batch_reward if i == cfg.test_length - 1 else batch_reward + cfg.gamma * q_next.max(1)[0].view(cfg.batch_size, 1)
                loss     = loss_func(q_eval, q_target)

                optimizer.zero_grad()
                loss.backward()
                Apply_Positive_Constraint(eval_net)
                optimizer.step()

                learn_step_counter += 1
                if learn_step_counter % cfg.q_network_iteration == 0:
                    target_net.load_state_dict(eval_net.state_dict())

        ### Validation ###
        if (j + 1) % cfg.validation_interval == 0:
            eval_net.eval()
            valid_theta = validation_theta
            valid_bias  = np.zeros((cfg.test_length, cfg.validation_size))

            state   = np.concatenate((
                np.zeros((cfg.input_size - 1, cfg.validation_size)),
                np.expand_dims(validation_initial_theta, axis=0),
            ))
            item_id = np.array([])

            for i in range(cfg.test_length):
                action = Choose_Action_Test(item_id, state)
                step_resp = validation_response_matrix[
                    np.arange(cfg.validation_size), action
                ]
                if i == 0:
                    item_id = action[np.newaxis, :]
                    resp = step_resp[np.newaxis, :]
                else:
                    item_id = np.concatenate((item_id, action[np.newaxis, :]))
                    resp = np.concatenate((resp, step_resp[np.newaxis, :]))

                theta_0  = np.zeros(cfg.validation_size)
                idx_full = np.sum(resp, axis=0) == resp.shape[0]
                idx_zero = np.sum(resp, axis=0) == 0
                idx_norm = np.bitwise_not(idx_full | idx_zero)
                theta_0[idx_full] = state[-1, idx_full] + (item_bank[:, 1].max() - state[-1, idx_full]) / 2
                theta_0[idx_zero] = state[-1, idx_zero] + (item_bank[:, 1].min() - state[-1, idx_zero]) / 2
                theta_0[idx_norm] = np.squeeze(MLE_TEST(item_bank[item_id[:, idx_norm]], resp[:, idx_norm]))

                if cfg.input_size > 1:
                    state = np.concatenate((state[-(cfg.input_size - 1):], theta_0[np.newaxis, :]))
                else:
                    state = theta_0[np.newaxis, :]
                valid_bias[i] = theta_0 - valid_theta

            step_valid = np.transpose(np.vstack((
                np.arange(1, cfg.test_length + 1),
                np.mean(valid_bias, axis=1),
                np.sqrt(np.mean(valid_bias ** 2, axis=1)),
                np.mean(abs(valid_bias), axis=1),
            )))
            result_valid = np.mean(step_valid[:, 1:], axis=0)
            if best_valid is None:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())
            elif result_valid[1] < best_valid[1]:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())

            eval_net.train()

    return best_state

In [18]:
def EVALUATE(cfg, theta_eval, initial_theta, response_matrix):
    with torch.no_grad():
        eval_net.eval()
        testing_size = len(theta_eval)

        state   = np.concatenate((
            np.zeros((cfg.input_size - 1, testing_size)),
            np.expand_dims(initial_theta, axis=0),
        ))
        item_id  = np.array([])
        dqn_step = np.zeros((1, 4))

        for i in range(cfg.test_length):
            action = Choose_Action_Test(item_id, state)
            step_resp = response_matrix[np.arange(testing_size), action]
            if i == 0:
                item_id = action[np.newaxis, :]
                resp = step_resp[np.newaxis, :]
            else:
                item_id = np.concatenate((item_id, action[np.newaxis, :]))
                resp = np.concatenate((resp, step_resp[np.newaxis, :]))

            theta_0  = np.zeros([1, testing_size])
            idx_full = np.sum(resp, axis=0) == resp.shape[0]
            idx_zero = np.sum(resp, axis=0) == 0
            idx_norm = np.bitwise_not(idx_full | idx_zero)
            theta_0[:, idx_full] = state[-1, idx_full] + (item_bank[:, 1].max() - state[-1, idx_full]) / 2
            theta_0[:, idx_zero] = state[-1, idx_zero] + (item_bank[:, 1].min() - state[-1, idx_zero]) / 2
            theta_0[:, idx_norm] = MLE_TEST(item_bank[item_id[:, idx_norm]], resp[:, idx_norm])

            if i == 0:
                theta = theta_0
            else:
                theta = np.concatenate((theta, theta_0))

            dqn_step = np.vstack([dqn_step, np.array([
                i + 1,
                np.mean(theta_0 - theta_eval),
                np.sqrt(np.mean((theta_0 - theta_eval) ** 2)),
                np.mean(abs(theta_0 - theta_eval)),
            ])])
            if cfg.input_size > 1:
                state = np.concatenate((state[-(cfg.input_size - 1):], theta_0))
            else:
                state = theta_0

        dqn_records = pd.DataFrame(
            {
                "userID": np.repeat(
                    np.arange(1, testing_size + 1), cfg.test_length
                ),
                "step": np.tile(
                    np.arange(1, cfg.test_length + 1), testing_size
                ),
                "itemID": (item_id + 1).T.reshape(-1).astype(np.int64),
                "resp": resp.T.reshape(-1).astype(np.int64),
                "theta_est": theta.T.reshape(-1),
                "bias": (theta - theta_eval).T.reshape(-1),
            }
        )
        dqn_summary = pd.DataFrame(
            dqn_step[1:], columns=["step", "Bias", "RMSE", "MAE"]
        )
        return dqn_records, dqn_summary

In [19]:
cfg = Config(
    # Network
    input_size    = 1,
    first_hidden  = 50,
    second_hidden = 30,
    dropout_rate  = 0.0,

    # Training
    test_length         = 10,
    gamma               = 0.1,
    memory_capacity     = 1000,
    epsilon             = 0.1,
    batch_size          = 128,
    q_network_iteration = 40,
    learning_rate       = 1e-3,
    training_size       = 1000,
    validation_size     = 200,
    validation_interval = 50,
    seed                = 20260430,
    validation_seed     = 20260431,
    test_seed           = 20260432,

    # Bank / prior
    bank_id   = 1,
    prior     = "uniform",  # "normal" | "uniform"
    n_items   = 100,
)

bank_dir = ROOT / "EXP_v2" / "data" / f"2PL_{cfg.n_items}items"

item_bank    = np.array(pd.read_csv(bank_dir / f"item_bank_uncor_{cfg.bank_id}.csv")[["a", "b"]])[:cfg.n_items]
action_space = item_bank.shape[0]

training_rng = np.random.default_rng(cfg.seed)
validation_rng = np.random.default_rng(cfg.validation_seed)
if cfg.prior == "normal":
    training_theta = training_rng.standard_normal(cfg.training_size)
    validation_theta = validation_rng.standard_normal(cfg.validation_size)
elif cfg.prior == "uniform":
    training_theta = training_rng.uniform(-3, 3, cfg.training_size)
    validation_theta = validation_rng.uniform(-3, 3, cfg.validation_size)
else:
    raise ValueError(f"Unsupported prior: {cfg.prior!r}. Use 'normal' or 'uniform'.")

validation_initial_theta = validation_rng.random(cfg.validation_size) - 0.5
a = item_bank[:, 0][np.newaxis, :]
b = item_bank[:, 1][np.newaxis, :]
probability = 1.0 / (
    1.0 + np.exp(-a * (validation_theta[:, np.newaxis] - b))
)
validation_response_matrix = (
    validation_rng.random(probability.shape) <= probability
).astype(np.int64)

theta_test_path = (
    ROOT / "EXP_v2" / "data" / "theta_true" / f"theta_true_{cfg.bank_id}.csv"
)
theta_test = pd.read_csv(theta_test_path)["x"].to_numpy(dtype=np.float64)
test_rng = np.random.default_rng(cfg.test_seed)
test_initial_theta = test_rng.random(len(theta_test)) - 0.5
test_probability = 1.0 / (
    1.0 + np.exp(-a * (theta_test[:, np.newaxis] - b))
)
test_response_matrix = (
    test_rng.random(test_probability.shape) <= test_probability
).astype(np.int64)

GAMMA_CANDIDATES = (0.0, 0.1, 0.3, 0.5, 0.7, 0.9)
EPSILON_CANDIDATES = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)

print(f"item bank       : {item_bank.shape}")
print(f"training theta  : {training_theta.shape}")
print(f"validation theta: {validation_theta.shape} (independent)")
print(f"test theta      : {theta_test.shape} ({theta_test_path})")
print(f"gamma candidates  : {GAMMA_CANDIDATES}")
print(f"epsilon candidates: {EPSILON_CANDIDATES}")
print(f"\nBase config:\n{cfg}")

item bank       : (100, 2)
training theta  : (1000,)
validation theta: (200,) (independent)
test theta      : (5000,) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v2/data/theta_true/theta_true_1.csv)
gamma candidates  : (0.0, 0.1, 0.3, 0.5, 0.7, 0.9)
epsilon candidates: (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)

Base config:
Config(input_size=1, first_hidden=50, second_hidden=30, dropout_rate=0.0, test_length=10, gamma=0.1, memory_capacity=1000, epsilon=0.1, batch_size=128, q_network_iteration=40, learning_rate=0.001, training_size=1000, validation_size=200, validation_interval=50, seed=20260430, validation_seed=20260431, test_seed=20260432, bank_id=1, prior='uniform', n_items=100)


In [20]:
research_rows = []
selected_state = None
selected_cfg = None
selected_mean_rmse = np.inf

for gamma in GAMMA_CANDIDATES:
    for epsilon in EPSILON_CANDIDATES:
        run_cfg = replace(cfg, gamma=gamma, epsilon=epsilon)
        set_seed(run_cfg.seed)

        eval_net = Net(
            run_cfg.input_size,
            run_cfg.first_hidden,
            run_cfg.second_hidden,
            action_space,
            run_cfg.dropout_rate,
        ).to(device)
        target_net = Net(
            run_cfg.input_size,
            run_cfg.first_hidden,
            run_cfg.second_hidden,
            action_space,
            run_cfg.dropout_rate,
        ).to(device)
        eval_net.initialize()
        target_net.initialize()

        best_state = TRAIN(
            run_cfg,
            training_theta,
            validation_theta,
            validation_initial_theta,
            validation_response_matrix,
        )
        if best_state is None:
            raise RuntimeError(
                "No checkpoint was selected. Increase training_size or "
                "lower validation_interval."
            )
        eval_net.load_state_dict(best_state)

        _, validation_summary = EVALUATE(
            run_cfg,
            validation_theta,
            validation_initial_theta,
            validation_response_matrix,
        )
        mean_rmse = float(validation_summary["RMSE"].mean())
        research_rows.append(
            {"gamma": gamma, "epsilon": epsilon, "mean_RMSE": mean_rmse}
        )
        if mean_rmse < selected_mean_rmse:
            selected_mean_rmse = mean_rmse
            selected_state = copy.deepcopy(best_state)
            selected_cfg = run_cfg
        print(
            f"gamma={gamma:.1f}, epsilon={epsilon:.1f}, "
            f"mean RMSE={mean_rmse:.6f}"
        )

research_results = pd.DataFrame(research_rows).sort_values(
    ["mean_RMSE", "gamma", "epsilon"], ignore_index=True
)
best = research_results.iloc[0]
if selected_state is None or selected_cfg is None:
    raise RuntimeError("No model was selected by the parameter search.")
eval_net.load_state_dict(selected_state)
print("\nBest hyperparameters on the independent validation set")
print(f"gamma        : {best['gamma']:.1f}")
print(f"epsilon      : {best['epsilon']:.1f}")
print(f"mean RMSE    : {best['mean_RMSE']:.6f}")
display(research_results)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / (
    f"dqn_{selected_cfg.prior}_2pl_"
    f"{selected_cfg.bank_id}_{selected_cfg.n_items}items_"
    f"epsilon_{selected_cfg.epsilon}_gamma_{selected_cfg.gamma}_"
    "research_param.pt"
)
torch.save(selected_state, model_path)
print(f"\nBest model saved to: {model_path}")

test_records, test_summary = EVALUATE(
    selected_cfg,
    theta_test,
    test_initial_theta,
    test_response_matrix,
)
test_mean_rmse = float(test_summary["RMSE"].mean())
print("\nIndependent test result using the selected model")
print(f"gamma        : {selected_cfg.gamma:.1f}")
print(f"epsilon      : {selected_cfg.epsilon:.1f}")
print(f"mean RMSE    : {test_mean_rmse:.6f}")
display(test_summary)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
result_stem = (
    f"2pl_{selected_cfg.bank_id}_"
    f"{selected_cfg.n_items}items_DQN_{selected_cfg.prior}_"
    f"epsilon_{selected_cfg.epsilon}_gamma_{selected_cfg.gamma}_"
    "research_param"
)
records_path = RESULTS_DIR / f"records_{result_stem}.csv"
summary_path = RESULTS_DIR / f"summary_{result_stem}.csv"
test_records.to_csv(records_path, index=False)
test_summary.to_csv(summary_path, index=False)
print(f"Saved records to: {records_path}")
print(f"Saved summary to: {summary_path}")

gamma=0.0, epsilon=0.0, mean RMSE=0.685807
gamma=0.0, epsilon=0.1, mean RMSE=0.636708
gamma=0.0, epsilon=0.2, mean RMSE=0.643000
gamma=0.0, epsilon=0.3, mean RMSE=0.599072
gamma=0.0, epsilon=0.4, mean RMSE=0.624170
gamma=0.0, epsilon=0.5, mean RMSE=0.636013
gamma=0.1, epsilon=0.0, mean RMSE=0.653122
gamma=0.1, epsilon=0.1, mean RMSE=0.663413
gamma=0.1, epsilon=0.2, mean RMSE=0.614426
gamma=0.1, epsilon=0.3, mean RMSE=0.603020
gamma=0.1, epsilon=0.4, mean RMSE=0.621754
gamma=0.1, epsilon=0.5, mean RMSE=0.616680
gamma=0.3, epsilon=0.0, mean RMSE=0.634096
gamma=0.3, epsilon=0.1, mean RMSE=0.653588
gamma=0.3, epsilon=0.2, mean RMSE=0.600169
gamma=0.3, epsilon=0.3, mean RMSE=0.608230
gamma=0.3, epsilon=0.4, mean RMSE=0.624284
gamma=0.3, epsilon=0.5, mean RMSE=0.612655
gamma=0.5, epsilon=0.0, mean RMSE=0.683974
gamma=0.5, epsilon=0.1, mean RMSE=0.613684
gamma=0.5, epsilon=0.2, mean RMSE=0.634593
gamma=0.5, epsilon=0.3, mean RMSE=0.610325
gamma=0.5, epsilon=0.4, mean RMSE=0.652436
gamma=0.5, 

,gamma,epsilon,mean_RMSE
0,0.0,0.3,0.599072
1,0.3,0.2,0.600169
2,0.1,0.3,0.603020
3,0.7,0.2,0.605290
4,0.3,0.3,0.608230
5,0.5,0.3,0.610325
6,0.3,0.5,0.612655
7,0.7,0.3,0.613150
8,0.5,0.1,0.613684
9,0.1,0.2,0.614426



Best model saved to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v2/models/dqn_uniform_2pl_1_100items_epsilon_0.3_gamma_0.0_research_param.pt

Independent test result using the selected model
gamma        : 0.0
epsilon      : 0.3
mean RMSE    : 0.575904


,step,Bias,RMSE,MAE
0,1.0,-0.199624,0.904895,0.729918
1,2.0,-0.022477,0.711666,0.563380
2,3.0,-0.006661,0.639860,0.509916
3,4.0,0.001926,0.595776,0.473677
4,5.0,0.011608,0.542312,0.431015
5,6.0,0.015697,0.511638,0.406592
6,7.0,0.013250,0.489597,0.390668
7,8.0,0.013631,0.469357,0.372784
8,9.0,0.007550,0.454999,0.361034
9,10.0,0.007647,0.438937,0.347025


Saved records to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v2/results/records_2pl_1_100items_DQN_uniform_epsilon_0.3_gamma_0.0_research_param.csv
Saved summary to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v2/results/summary_2pl_1_100items_DQN_uniform_epsilon_0.3_gamma_0.0_research_param.csv
